In [1]:
## define data 
folder_path = "./raw/"
## loop through all csv files in folder raw and merge them into one dataframe
import os
import pandas as pd
all_files = [f for f in os.listdir(folder_path) if f.endswith('.csv')]
df_list = []
for file in all_files:
    file_path = os.path.join(folder_path, file)
    df_temp = pd.read_csv(file_path)
    df_list.append(df_temp)
df = pd.concat(df_list, ignore_index=True)




## Penggabungan kolom teks menjadi narasi, hapus duplikat, dan simpan hasil bersih.
Langkah:

Load & Gabung: Isi NaN diganti string kosong dan gabungkan kolom yang tersedia.
Cek Duplikat: Hitung dan lihat baris duplikat.
Hapus Duplikat: Drop duplikat dari DataFrame.
Simpan: Simpan ke ./processed/data_bersih_<timestamp>.csv.

In [ ]:

import pandas as pd



nama_kolom = 'Isi Postingan'

cols_to_merge = []

if 'Isi Tweet' in df.columns:
    df['Isi Tweet'] = df['Isi Tweet'].fillna('')
    cols_to_merge.append('Isi Tweet')

if 'Isi Postingan' in df.columns:
    df['Isi Postingan'] = df['Isi Postingan'].fillna('')
    cols_to_merge.append('Isi Postingan')

if cols_to_merge:
    # Menggabungkan nilai dari kolom-kolom yang ada menjadi satu string yang dipisahkan spasi
    df['narasi'] = df[cols_to_merge].astype(
        str).agg(' '.join, axis=1).str.strip()

    # Menghapus kolom asli yang sudah digabungkan (Isi Tweet dan/atau Isi Postingan)
    df.drop(columns=cols_to_merge, inplace=True)
else:
    # Jika tidak ada kedua kolom tersebut, buat kolom narasi kosong
    df['narasi'] = ''



# 2. Mengecek jumlah duplikasi
jumlah_duplikat = df.duplicated().sum()
print(f"Jumlah baris duplikat: {jumlah_duplikat}")

baris_duplikat = df[df.duplicated()]
# display(baris_duplikat)

# 3. Menghapus duplikasi

df.drop_duplicates(inplace=True)

# 4. Menyimpan kembali ke file CSV yang baru (atau menimpa yang lama)

timestamp = pd.Timestamp.now().strftime("%Y%m%d_%H%M%S")

nama_file_bersih = 'data_bersih_' + timestamp + '.csv'

##check apakah ada folder process, jika tidak ada buat folder process
import os
if not os.path.exists('./processed'):
    os.makedirs('./processed')


df.to_csv('./processed/' + nama_file_bersih, index=False)

print("Penghapusan duplikat selesai dan file baru telah disimpan.")

baris_duplikat.head(10)

Jumlah baris duplikat: 92
Penghapusan duplikat selesai dan file baru telah disimpan.


,id,narasi
1058,NaN,"Pada hari Sabtu, 04 April 2026 Puskesmas Medon..."
1082,NaN,Cek kesehatan gratis\n1. Pendaftaran dengan me...
1158,NaN,PUSKESMAS SEHAT \nKegiatan Cek Kesehatan Grati...
1197,NaN,CKG ( Cek kesehatan gratis ) \nPasar batang 18...
1211,NaN,"Day 6 !!! \nRabu, 8 April 2026.\nCKG ( Cek Kes..."
1286,NaN,Nda pusing sayang ? \nCek Kesehatan Gratis ( C...
1288,NaN,Giat Cek Kesehatan Gratis (CKG) oleh Puskesmas...
1293,NaN,SEMANGAT SEHAT BANTEN 2026 \nWujudkan gaya hid...
1354,NaN,Cek Kesehatan Gratis (CKG) Anak Sekolah adalah...
1359,NaN,Dewan Pimpinan Cabang (DPC) PDI Perjuangan Kab...


## Pengecekan row dengan tagar saja

In [3]:

nama_kolom = 'narasi'

# 1. Hapus semua tagar (misal: #tagar) dan hapus spasi berlebih di awal/akhir
# r'#\w+' adalah pola regex untuk mencocokkan kata yang dimulai dengan '#'
sisa_teks = df[nama_kolom].str.replace(r'#\w+', '', regex=True).str.strip()

# 2. Buat kondisi untuk mengecek baris mana yang sisa teksnya kosong
# Jika kosong (''), berarti baris aslinya hanya berisi tagar (dan/atau spasi)
kondisi_hanya_tagar = sisa_teks == ''

# (Opsional) Tampilkan berapa banyak dan apa saja datanya
jumlah_hanya_tagar = kondisi_hanya_tagar.sum()
print(f"Jumlah baris yang HANYA berisi tagar: {jumlah_hanya_tagar}")
if jumlah_hanya_tagar > 0:
    display(df[kondisi_hanya_tagar].head())

# 3. Filter dataframe: Simpan data yang TIDAK hanya berisi tagar (pakai simbol ~)
df = df[~kondisi_hanya_tagar]

print(f"Sisa total data sekarang: {len(df)}")

Jumlah baris yang HANYA berisi tagar: 1064


,id,narasi
7,NaN,#fbprofesional
11,NaN,#CekKesehatanGratis #CKG #SehatBersama #puskes...
20,NaN,#jangkauanlas
23,NaN,#TabananEraBaru
30,NaN,#anaksekolah


Sisa total data sekarang: 1827


## Mengambil baris yang hanya mengandung policy_keyword

In [ ]:
import pandas as pd
import re


# Keywords related to Prabowo's policies
policy_keywords = [
    'prabowo','ckg', 'kesehatan gratis','cek kesehatan gratis', 'cek kesehatan gratis 2025', 'cek kesehatan gratis 2026'
]

# Keywords/patterns indicating news
news_patterns = [
    # r'http\S+', r'www\.\S+', r'\.com', r'\.co\.id', r'\.tv',
    # r'baca di', r'simak', r'selengkapnya', r'sc:', r'sumber:',
    # r'#news\b', r'info bansos', r'reporter', r'kata prabowo', r'mengatakan', r'Follow @\w+', r'via @\w+', r'berita', r'update', r'breaking news', r'headline', r'laporan', r'kabar terbaru', r'follow'
]


ckg_patterns = [
    r'cek kesehatan gratis', r'ckg', r'prabowo cek kesehatan gratis', r'#cekkesehatangratis\b', r"cek kesehatan gratis 2025", r"cek kesehatan gratis 2026"

]


def is_news(text):
    text_lower = str(text).lower()
    for pattern in news_patterns:
        if re.search(pattern, text_lower):
            return True
    return False


def is_policy(text):
    text_lower = str(text).lower()
    for kw in policy_keywords:
        if kw in text_lower:
            return True
    return False


def is_ckg(text):
    text_lower = str(text).lower()
    for pattern in ckg_patterns:
        if re.search(pattern, text_lower):
            return True
    return False



# Filter for policy
df_policy = df[df['narasi'].apply(is_policy)].copy()
# filter for CKG
df_policy = df_policy[df_policy['narasi'].apply(is_ckg)].copy()

print(f"Total data: {len(df)}")
print(f"Total data related to policy & CKG: {len(df_policy)}")

# Buat teks menjadi huruf kecil
df_policy['narasi'] = df_policy['narasi'].str.lower()

# Simpan semua data (opini + berita) ke CSV
timestamp = pd.Timestamp.now().strftime("%Y%m%d_%H%M%S")


## rapikan data hilangkan /n atau / atau spasi berlebih dan masih banyak lagi yang harus dihapus, buat fungsi untuk membersihkan teks double comma, double space, dan karakter khusus lainnya
def bersihkan_teks(teks):
    if pd.isna(teks):
        return ''

    # 1. Hapus karakter khusus pembatas baris (\n, \r, \t)
    teks = re.sub(r'[\n\r\t]+', ' ', teks)

    # 2. Hapus simbol aneh, HANYA pertahankan huruf, angka, spasi, dan tanda baca umum (.,!?;:-)
    teks = re.sub(r'[^\w\s.,!?;:\-]', ' ', teks)

    # 3. Hapus duplikasi tanda baca (misal: ",," menjadi ",", ".." menjadi ".", "!!!" menjadi "!")
    # r'([.,!?;:\-])\1+' akan mencari tanda baca yang diulang dan menyisakannya menjadi satu saja
    teks = re.sub(r'([.,!?;:\-])\1+', r'\1', teks)

    # 4. Hapus spasi berlebih dan spasi awal/akhir
    teks = re.sub(r'\s+', ' ', teks).strip()

    return teks
df_policy['narasi'] = df_policy['narasi'].apply(bersihkan_teks)
output_file = './processed/opini cek kesehatan gratis_' + timestamp + '.csv'
# Simpan df_policy, bukan df_opinion
df_policy[['narasi']].to_csv(output_file, index=False)

print("Saved to", output_file)
print(df_policy['narasi'].head(10).tolist())

Total data: 1745
Total data related to policy & CKG: 549
Saved to ./processed/opini cek kesehatan gratis_20260416_134014.csv
['mari kita datangi fasilitas kesehatan yang ada untuk cek kesehatan gratis', 'jangan lewatkan kesempatan ini. bapa mama to o te o dong mari su datang . jang tunggu sakit do baru periksa, lebih baik cek dari sekarang. mari datang cek kesehatan gratis, sehat tu penting! with maria isabela and promkes dinkes rote ndao .', 'cek kesehatan gratis ckg anak sekolah adalah program puskesmas untuk mendeteksi dini risiko kesehatan siswa- siswi 7-17 tahun di sekolah. layanan ini meliputi pemeriksaan tinggi badan, berat badan, tensi darah, telinga dan gigi.', 'personil kompi kesehatan yonif tp 834 wm melaksanakan kegiatan ckg cek kesehatan gratis sebagai bentuk kepedulian terhadap kesehatan masyarakat di wilayah sekitar. kegiatan ini berlangsung di pustu aeramo dengan penuh semangat dan dedikasi guna memastikan kondisi kesehatan warga tetap terjaga. selain itu, personil komp

## Melihat data yang terbaru pada file processed

In [7]:
import pandas as pd
import glob
import os

# Mencari semua file CSV yang namanya diawali dengan "opini cek kesehatan gratis_"
list_files = glob.glob('./processed/opini cek kesehatan gratis_*.csv')

if list_files:
    # Mengambil file yang paling baru berdasarkan waktu modifikasi (atau bisa juga max() karena format timestamp)
    recent_file = max(list_files, key=os.path.getmtime)
    print(f"Loading file terbaru: {recent_file}")

    # Membuka file terbaru
    opini = pd.read_csv(recent_file)
else:
    print("File tidak ditemukan.")
    
opini.head()

Loading file terbaru: ./processed\opini cek kesehatan gratis_20260416_134014.csv


,narasi
0,mari kita datangi fasilitas kesehatan yang ada...
1,jangan lewatkan kesempatan ini. bapa mama to o...
2,cek kesehatan gratis ckg anak sekolah adalah p...
3,personil kompi kesehatan yonif tp 834 wm melak...
4,assalamu alaikum. warga segarau dan sungai pug...
